# BBC News Business Subcategory Classifier

This notebook implements the full subcategory classification pipeline for the **Business** category of the BBC News dataset..

**Pipeline Overview:**
- Phase 1 — Data Ingestion
- Phase 2 — Text Cleaning
- Phase 3 — Vectorization (Jina AI)
- Phase 4 — UMAP + HDBSCAN Clustering
- Phase 5 — c-TF-IDF + KeyBERT Labeling
- Phase 5B — Verification Block
- Phase 6 — AI Subcategory Naming
- Phase 8 — Manual Noise Correction
- Phase 9 — Named Entity Recognition (NER) with April Events Extraction


## Project Overview
This notebook classifies 510 BBC Business articles into meaningful subcategories, Beyond subcategory classification, the pipeline also performs Named Entity Recognition (NER) to extract media personalities and their roles, and identifies any events that took place or were scheduled to take place in April. using Jina Embeddings v5, UMAP dimensionality reduction, and HDBSCAN density-based clustering.

**Author:** BBC NLP Project 2
**Dataset:** 510 BBC Business Articles
**Model:** Jina Embeddings v5 (text-small) with clustering adapter

---

## Phase 1: Data Ingestion & Structural Partitioning
In this phase we load all 510 business articles from the `./business` folder into a structured DataFrame. Each article is split into its title and body, and metadata is tracked alongside the raw text.

In [1]:
# ============================================================
# PHASE 1: DATA INGESTION & STRUCTURAL PARTITIONING
# ============================================================

import os
import pandas as pd

# 1. Define the path to the business folder
folder_path = './business'

# 2. Initialize our data collection list
all_articles = []

# 3. Loop through each .txt file in the business folder
for filename in sorted(os.listdir(folder_path)):
    if filename.endswith('.txt'):
        file_path = os.path.join(folder_path, filename)

        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read().strip()

        # 4. Split the file into title (first line) and body (rest)
        lines = content.split('\n')
        title = lines[0].strip()
        body = '\n'.join(lines[1:]).strip()

        # 5. Track metadata alongside raw text
        all_articles.append({
            'file_name': filename,
            'article_id': int(filename.replace('.txt', '')),
            'title': title,
            'raw_text': body,
            'article_length': len(body.split())
        })

# 6. Load into a structured DataFrame
df = pd.DataFrame(all_articles)

# 7. Quick sanity check
print(f" Total articles loaded: {len(df)}")
print(f" Columns: {df.columns.tolist()}")
print(f"\n Sample:")
df.head()

 Total articles loaded: 510
 Columns: ['file_name', 'article_id', 'title', 'raw_text', 'article_length']

 Sample:


,file_name,article_id,title,raw_text,article_length
0,001.txt,1,Ad sales boost Time Warner profit,Quarterly profits at US media giant TimeWarner...,415
1,002.txt,2,Dollar gains on Greenspan speech,The dollar has hit its highest level against t...,379
2,003.txt,3,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yuko...,258
3,004.txt,4,High fuel prices hit BA's profits,British Airways has blamed high fuel prices fo...,400
4,005.txt,5,Pernod takeover talk lifts Domecq,Shares in UK drinks and food firm Allied Domec...,260


## Phase 2: The Refinement Layer (Data Cleaning)

In this phase we clean all business articles to prepare them for vectorization.
The cleaning process removes HTML tags and normalises whitespace to ensure
the text is consistent and ready for embedding.

In [5]:
# ============================================================
# PHASE 2: DATA CLEANING — BUSINESS
# ============================================================
import re
import pandas as pd

def clean_article(text):

    # 1. HTML REMOVAL - confirmed present in raw BBC data
    text = re.sub(r'<[^>]+>', '', text)

    # 2. WHITESPACE CLEANUP
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()

    return text

df = pd.read_csv('business_subcategories_output.csv')

if 'raw_text' not in df.columns:
    df = df.rename(columns={'cleaned_text': 'raw_text'})

df['cleaned_text'] = df['raw_text'].apply(clean_article)
df = df.drop(columns=['raw_text'])

df.to_csv('business_subcategories_output.csv', index=False)

print(f"Total articles: {len(df)}")
print(f"\nBusiness cleaning complete!")

Total articles: 510

Business cleaning complete!


## Phase 3: Vectorization Layer (Jina-v5 Intelligence)
In this phase we convert each cleaned article into a 1024-dimensional embedding vector using Jina AI's latest embedding model with the clustering adapter. Articles are processed in batches of 20 with auto-retry on rate limit errors.

In [3]:
# ============================================================
# PHASE 3: THE VECTORIZATION LAYER (JINA-V5) - RATE LIMIT FIX
# ============================================================

import requests
import numpy as np
import time
import os

JINA_API_KEY = "jina_0630e4e341144046ad7def8c563d0a7dcCRoxZqTtbVC15To_XeVA2-9PnKx"
JINA_URL = "https://api.jina.ai/v1/embeddings"

HEADERS = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {JINA_API_KEY}"
}

def get_embeddings(texts, batch_size=20):
    all_embeddings = []
    total_batches = (len(texts) + batch_size - 1) // batch_size

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        batch_num = (i // batch_size) + 1

        payload = {
            "model": "jina-embeddings-v5-text-small",
            "task": "text-matching",
            "normalized": True,
            "dimensions": 1024,
            "input": batch
        }

        success = False
        retries = 3

        while not success and retries > 0:
            try:
                response = requests.post(JINA_URL, headers=HEADERS, json=payload)
                response.raise_for_status()
                data = response.json()

                batch_embeddings = [item['embedding'] for item in data['data']]
                all_embeddings.extend(batch_embeddings)

                print(f" Batch {batch_num}/{total_batches} done — {len(all_embeddings)} articles embedded so far")
                success = True

                time.sleep(3)

            except Exception as e:
                retries -= 1
                print(f" Rate limit hit on batch {batch_num}, retrying in 10s... ({retries} retries left)")
                time.sleep(10)

        if not success:
            print(f" Failed on batch {batch_num} after all retries")
            break

    return np.array(all_embeddings)

# Only embed if embeddings don't already exist
if os.path.exists('embeddings.npy'):
    print(" Embeddings already exist — skipping API call")
    embeddings = np.load('embeddings.npy')
else:
    # Run on ALL 510 articles using title + content combined
    texts = (df['title'] + " " + df['cleaned_text']).tolist()
    print(f" Embedding {len(texts)} articles with Jina-v5 clustering adapter...\n")

    embeddings = get_embeddings(texts)

    # Save embeddings to disk permanently
    np.save('embeddings.npy', embeddings)
    print(f"\n Embedding complete!")

print(f" Embedding matrix shape: {embeddings.shape}")
print(f"   → {embeddings.shape[0]} articles x {embeddings.shape[1]} dimensions")

 Embeddings already exist — skipping API call
 Embedding matrix shape: (510, 1024)
   → 510 articles x 1024 dimensions


## Dependencies
Installing required libraries for dimensionality reduction and clustering — UMAP and HDBSCAN.

In [4]:
!pip install umap-learn hdbscan


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Phase 4: Dimensionality Reduction & Clustering
In this phase we reduce the 1024-dimensional embeddings to 3 dimensions using UMAP while preserving thematic relationships between articles. HDBSCAN then discovers the natural number of subcategories automatically without any pre-set cluster count. Results are saved to disk permanently to ensure reproducibility.

In [5]:
# ============================================================
# PHASE 4: DIMENSIONALITY REDUCTION & CLUSTERING
# ============================================================

import umap
import hdbscan
import numpy as np
import os

# Load embeddings from disk if available
if os.path.exists('embeddings.npy'):
    print(" Loading saved embeddings...")
    embeddings = np.load('embeddings.npy')
    print(f" Embeddings shape: {embeddings.shape}")

# Check if we already have saved cluster results
if os.path.exists('cluster_labels.npy') and os.path.exists('reduced_embeddings.npy'):
    
    print(" Loading saved cluster results...")
    cluster_labels = np.load('cluster_labels.npy')
    reduced_embeddings = np.load('reduced_embeddings.npy')
    confidence_scores = np.load('confidence_scores.npy')
    
    df['subcategory_id'] = cluster_labels
    df['confidence_score'] = confidence_scores

    n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
    n_noise = list(cluster_labels).count(-1)

    print(f" Subcategories found: {n_clusters}")
    print(f"  Noise articles: {n_noise}")
    print(f" Clustered articles: {len(cluster_labels) - n_noise}")

else:
    
    # 1. UMAP
    print(" Running UMAP dimensionality reduction...")
    reducer = umap.UMAP(
        n_components=3,
        n_neighbors=15,
        min_dist=0.0,
        metric='cosine',
        random_state=42
    )
    reduced_embeddings = reducer.fit_transform(embeddings)
    print(f" UMAP complete! Shape: {reduced_embeddings.shape}\n")

    # 2. HDBSCAN
    print(" Running HDBSCAN clustering...")
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=8,
        min_samples=4,
        metric='euclidean',
        cluster_selection_method='eom'
    )
    cluster_labels = clusterer.fit_predict(reduced_embeddings)
    confidence_scores = clusterer.probabilities_
    confidence_scores[cluster_labels == -1] = 0.0

    df['subcategory_id'] = cluster_labels
    df['confidence_score'] = confidence_scores

    # Save to disk permanently
    np.save('cluster_labels.npy', cluster_labels)
    np.save('reduced_embeddings.npy', reduced_embeddings)
    np.save('confidence_scores.npy', confidence_scores)
    print(" Results saved to disk permanently!")

    n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
    n_noise = list(cluster_labels).count(-1)

    print(f" HDBSCAN complete!")
    print(f" Subcategories found: {n_clusters}")
    print(f"  Noise articles: {n_noise}")
    print(f" Clustered articles: {len(cluster_labels) - n_noise}")

# Cluster distribution
print(f"\n Articles per subcategory:")
print(df['subcategory_id'].value_counts().sort_index())

 Loading saved embeddings...
 Embeddings shape: (510, 1024)
 Loading saved cluster results...
 Subcategories found: 18
  Noise articles: 71
 Clustered articles: 439

 Articles per subcategory:
subcategory_id
-1     71
 0     31
 1     44
 2     11
 3     10
 4     16
 5     26
 6     10
 7     11
 8     82
 9     10
 10    20
 11     8
 12    49
 13    20
 14    42
 15    21
 16    13
 17    15
Name: count, dtype: int64


## Phase 5: Thematic Labeling (c-TF-IDF)
In this phase we extract the most unique and discriminative keywords for each cluster using c-TF-IDF. This compares word frequencies across clusters to surface words that are characteristic of each subcategory rather than just globally common words.

In [6]:
# ============================================================
# PHASE 5: THEMATIC LABELING (c-TF-IDF)
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import numpy as np

# 1. Separate clustered articles (exclude noise label -1)
clustered_df = df[df['subcategory_id'] != -1].copy()

# 2. Group all articles in each cluster into one big document
cluster_documents = clustered_df.groupby('subcategory_id')['cleaned_text'].apply(
    lambda texts: ' '.join(texts)
).reset_index()

# 3. c-TF-IDF — Compare word frequencies ACROSS clusters
# This finds words that are unique to each cluster (not just common everywhere)
vectorizer = TfidfVectorizer(
    max_features=10000,
    stop_words='english',
    ngram_range=(1, 2),    # Include bigrams like "stock market", "interest rate"
    min_df=1
)

tfidf_matrix = vectorizer.fit_transform(cluster_documents['cleaned_text'])
feature_names = vectorizer.get_feature_names_out()

# 4. Extract top keywords per cluster
def get_top_keywords(tfidf_matrix, feature_names, n=10):
    top_keywords = []
    for i in range(tfidf_matrix.shape[0]):
        row = tfidf_matrix[i].toarray()[0]
        top_indices = row.argsort()[-n:][::-1]
        keywords = [feature_names[idx] for idx in top_indices]
        top_keywords.append(keywords)
    return top_keywords

top_keywords = get_top_keywords(tfidf_matrix, feature_names, n=10)

# 5. Display keywords per cluster
print(" Top keywords per subcategory:\n")
for i, row in cluster_documents.iterrows():
    cluster_id = row['subcategory_id']
    keywords = top_keywords[i]
    print(f"Cluster {cluster_id:2d} | {', '.join(keywords)}")

# 6. Add top 3 keywords as semantic label to DataFrame
cluster_documents['semantic_label'] = [
    ' | '.join(kws[:3]) for kws in top_keywords
]

# 7. Map labels back to main DataFrame
label_map = dict(zip(cluster_documents['subcategory_id'], cluster_documents['semantic_label']))
df['semantic_label'] = df['subcategory_id'].map(label_map)
df['semantic_label'] = df['semantic_label'].fillna('Noise')

print(f"\n Semantic labels assigned!")
print(f"\n Cluster ID → Semantic Label:")
for cluster_id, label in sorted(label_map.items()):
    print(f"  Cluster {cluster_id:2d} → {label}")

 Top keywords per subcategory:

Cluster  0 | gm, fiat, car, bmw, said, rover, sales, nissan, mercedes, year
Cluster  1 | yukos, russian, gazprom, oil, rosneft, yugansk, said, court, russia, khodorkovsky
Cluster  2 | mortgage, housing, house prices, said, house, housing market, prices, halifax, market, lending
Cluster  3 | drug, drugs, patients, vioxx, said, phytopharm, johnson, gw, merck, firm
Cluster  4 | sri, lanka, sri lanka, said, thailand, disaster, damage, indonesia, tourism, usd
Cluster  5 | oil, crude, said, usd, prices, cairn, barrel, opec, fuel, production
Cluster  6 | japan, economy, japanese, recession, growth, year, economic, japan economy, takenaka, exports
Cluster  7 | age, pension, pensions, axa, said, gbp, life, retirement, employers, parents
Cluster  8 | said, economy, economic, growth, rate, year, rates, unemployment, spending, sales
Cluster  9 | china, cuba, 2004, year, economy, usd, exports, said, remittances, trade
Cluster 10 | dollar, turkey, said, usd, deficit, 

## KeyBERT Dependencies
Installing KeyBERT for article-level keyword extraction to complement the cluster-level c-TF-IDF analysis.

In [7]:
!pip install keybert


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Phase 5b: Article-Level Keyword Extraction (KeyBERT)
In this phase we extract article-level keywords using KeyBERT. Unlike c-TF-IDF which operates at the cluster level, KeyBERT extracts keywords that best represent each individual article — providing a richer and more explainable output.

In [8]:
# ============================================================
# PHASE 5B: KEYBERT + CLUSTER VERIFICATION
# ============================================================
from keybert import KeyBERT
import warnings
warnings.filterwarnings('ignore')

# 1. KeyBERT keyword extraction
print(" Loading KeyBERT model...")
kw_model = KeyBERT()
print(" KeyBERT model loaded!\n")

print(" Extracting keywords for each article...")
def extract_keywords(text, n=5):
    try:
        keywords = kw_model.extract_keywords(
            str(text),
            keyphrase_ngram_range=(1, 2),
            stop_words='english',
            top_n=n
        )
        return ', '.join([kw[0] for kw in keywords])
    except:
        return ''

df['article_keywords'] = df['cleaned_text'].apply(extract_keywords)
print(f" Keywords extracted for all {len(df)} articles!\n")

# 2. Cluster verification — read before naming
print("=" * 70)
print("      CLUSTER VERIFICATION — READ BEFORE NAMING")
print("=" * 70)

for cluster_id in sorted(df[df['subcategory_id'] != -1]['subcategory_id'].unique()):
    cluster_articles = df[df['subcategory_id'] == cluster_id]
    
    print(f"\n{'='*70}")
    print(f"CLUSTER {cluster_id} — {len(cluster_articles)} articles")
    print(f"{'='*70}")
    
    # Sample articles — focus on cleaned_text
    samples = cluster_articles.sample(min(3, len(cluster_articles)), random_state=42)
    for i, (_, row) in enumerate(samples.iterrows(), 1):
        print(f"\n Title {i}: {row['title']}")
        print(f" Keywords: {row['article_keywords']}")
        snippet = ' '.join(str(row['cleaned_text']).split()[:80])
        print(f" Text: {snippet}...")
    
    print(f"\n  WHAT LABEL SHOULD THIS CLUSTER GET?")
    print(f"{'='*70}")

 Loading KeyBERT model...
 KeyBERT model loaded!

 Extracting keywords for each article...
 Keywords extracted for all 510 articles!

      CLUSTER VERIFICATION — READ BEFORE NAMING

CLUSTER 0 — 31 articles

 Title 1: GM issues 2005 profits warning
 Keywords: sales gm, general motors, 2005 gm, gm, gm brands
 Text: General Motors has warned that it expects earnings this year be lower than in 2004. The world's biggest car maker is grappling with losses in its European business, and weak US sales. GM said higher healthcare costs in North America, and lower profits at its financial services subsidiary would hurt its performance in 2005. GM said it expects to meet its 2004 earnings targets "despite a tough competitive environment". GM, whose brands include Buick, Cadillac and Chevrolet in the US...

 Title 2: GM pays $2bn to evade Fiat buyout
 Keywords: fiat shares, fiat gm, fiat deal, firm fiat, fiat 1bn
 Text: General Motors of the US is to pay Fiat 1.55bn euros (USD 2bn; GBP 1.1bn) to ge

## Phase 6: Manual Label Assignment & Final Output
In this phase we assign professional subcategory names to each cluster using AI-generated labels based on the top c-TF-IDF keywords. The final output DataFrame is built and saved to CSV with all required columns.

In [ ]:
# ============================================================
## Phase 6: Manual Label Assignment & Final Output
# ============================================================

# 1. AI-generated subcategory names for 18 clusters
ai_label_map = {
    0:  "Automotive Industry",
    1:  "Crude Oil Markets",
    2:  "Housing Market",
    3:  "Pharmaceuticals",
    4:  "Disaster Economics",
    5:  "Uncategorised",
    6:  "Asian Economics",
    7:  "Pension Funds",
    8:  "Macroeconomic Indicators",
    9:  "International Trade",
    10: "Foreign Exchange",
    11: "Food Industry",
    12: "Corporate Governance",
    13: "Aviation Industry",
    14: "Telecommunications",
    15: "Financial Markets",
    16: "Sports Business",
    17: "Beverages Industry",
    -1: "Uncategorised"
}

# 2. Apply confidence scores
df['confidence_score'] = confidence_scores
df.loc[df['subcategory_id'] == -1, 'confidence_score'] = 0.0

# 3. Apply AI labels
df['semantic_label'] = df['subcategory_id'].map(ai_label_map)

# 4. Build final output DataFrame cleanly
final_df = df[[
    'file_name',
    'article_id',
    'title',
    'cleaned_text',
    'subcategory_id',
    'semantic_label',
    'confidence_score',
    'article_keywords'
]].copy().reset_index(drop=True)

# 5. Save to CSV
final_df.to_csv('business_subcategories_output.csv', index=False)

# 6. Summary
print("=" * 55)
print("      BUSINESS CLASSIFIER — FINAL RESULTS")
print("=" * 55)
print(f" Total Articles:        {len(final_df)}")
print(f" Subcategories Found:   {final_df[final_df['semantic_label'] != 'Uncategorised']['semantic_label'].nunique()}")
print(f" Clustered Articles:    {len(final_df[final_df['semantic_label'] != 'Uncategorised'])}")
print(f" Uncategorised Articles:{len(final_df[final_df['semantic_label'] == 'Uncategorised'])}")
print(f" Avg Confidence:        {final_df[final_df['semantic_label'] != 'Uncategorised']['confidence_score'].mean():.2f}")
print("=" * 55)
print(f"\n Label Distribution:")
print(final_df['semantic_label'].value_counts())
print(f"\n Saved to business_subcategories_output.csv")

      BUSINESS CLASSIFIER — FINAL RESULTS
 Total Articles:        510
 Subcategories Found:   17
 Clustered Articles:    413
 Uncategorised Articles:97
 Avg Confidence:        0.88

 Label Distribution:
semantic_label
Uncategorised               97
Macroeconomic Indicators    82
Corporate Governance        49
Crude Oil Markets           44
Telecommunications          42
Automotive Industry         31
Financial Markets           21
Aviation Industry           20
Foreign Exchange            20
Disaster Economics          16
Beverages Industry          15
Sports Business             13
Housing Market              11
Pension Funds               11
Pharmaceuticals             10
International Trade         10
Asian Economics             10
Food Industry                8
Name: count, dtype: int64

 Saved to business_subcategories_output.csv


## Phase 7: Noise Article Review
This phase reviews all articles assigned to the noise cluster (`subcategory_id = -1`) by HDBSCAN. Each noise article is printed with its ID, title, keywords, and a short text snippet to support manual inspection and informed reassignment in the next phase.

In [10]:
# View all uncategorised articles
uncategorised_df = final_df[final_df['semantic_label'] == 'Uncategorised'][['article_id', 'title', 'cleaned_text']].reset_index(drop=True)

for _, row in uncategorised_df.iterrows():
    print(f"ID: {row['article_id']}")
    print(f"Title: {row['title']}")
    print(f"Text: {row['cleaned_text'][:150]}")
    print("-" * 60)

ID: 9
Title: Ethiopia's crop production up 24%
Text: Ethiopia produced 14.27 million tonnes of crops in 2004, 24% higher than in 2003 and 21% more than the average of the past five years, a report says. 
------------------------------------------------------------
ID: 10
Title: Court rejects $280bn tobacco case
Text: A US government claim accusing the country's biggest tobacco companies of covering up the effects of smoking has been thrown out by an appeal court. T
------------------------------------------------------------
ID: 12
Title: Indonesians face fuel price rise
Text: Indonesia's government has confirmed it is considering raising fuel prices by as much as 30%. Millions of Indonesians use kerosene for basic cooking, 
------------------------------------------------------------
ID: 14
Title: Telegraph newspapers axe 90 jobs
Text: The Daily and Sunday Telegraph newspapers are axing 90 journalist jobs - 17% of their editorial staff. The Telegraph Group says the cuts are needed to


## Phase 8: Manual Noise Correction
Noise articles reviewed in Phase 7 are manually reassigned here based on their content, keywords, and title. Each article is mapped to the most appropriate subcategory and the updated labels are saved back to the output CSV.

In [11]:
# ============================================================
# PHASE 8: MANUAL NOISE CORRECTION
# ============================================================
import pandas as pd

# Load the CSV
df_final = pd.read_csv('business_subcategories_output.csv')

# Noise assignments grouped by subcategory
noise_assignments = {
    # Macroeconomics (19)
    30: "Macroeconomics", 33: "Macroeconomics", 41: "Macroeconomics",
    55: "Macroeconomics", 93: "Macroeconomics", 154: "Macroeconomics",
    159: "Macroeconomics", 204: "Macroeconomics", 213: "Macroeconomics",
    245: "Macroeconomics", 249: "Macroeconomics", 328: "Macroeconomics",
    330: "Macroeconomics", 371: "Macroeconomics", 412: "Macroeconomics",
    413: "Macroeconomics", 423: "Macroeconomics", 457: "Macroeconomics",
    488: "Macroeconomics",

    # Energy Markets (11)
    12: "Energy Markets", 28: "Energy Markets", 45: "Energy Markets",
    138: "Energy Markets", 144: "Energy Markets", 152: "Energy Markets",
    270: "Energy Markets", 351: "Energy Markets", 381: "Energy Markets",
    472: "Energy Markets", 502: "Energy Markets",

    # Corporate Earnings (12)
    42: "Corporate Earnings", 50: "Corporate Earnings", 53: "Corporate Earnings",
    111: "Corporate Earnings", 174: "Corporate Earnings", 207: "Corporate Earnings",
    221: "Corporate Earnings", 250: "Corporate Earnings", 288: "Corporate Earnings",
    310: "Corporate Earnings", 384: "Corporate Earnings", 396: "Corporate Earnings",

    # Mergers Acquisitions (10)
    21: "Mergers Acquisitions", 208: "Mergers Acquisitions",
    325: "Mergers Acquisitions", 341: "Mergers Acquisitions",
    366: "Mergers Acquisitions", 447: "Mergers Acquisitions",
    451: "Mergers Acquisitions", 470: "Mergers Acquisitions",
    485: "Mergers Acquisitions", 510: "Mergers Acquisitions",

    # Trade Policy (8)
    65: "Trade Policy", 67: "Trade Policy", 75: "Trade Policy",
    178: "Trade Policy", 320: "Trade Policy", 424: "Trade Policy",
    446: "Trade Policy", 478: "Trade Policy",

    # Corporate Governance (6)
    36: "Corporate Governance", 215: "Corporate Governance",
    282: "Corporate Governance", 397: "Corporate Governance",
    486: "Corporate Governance", 494: "Corporate Governance",

    # Oil Exploration (4)
    198: "Oil Exploration", 307: "Oil Exploration",
    313: "Oil Exploration", 433: "Oil Exploration",

    # Emerging Markets (3)
    18: "Emerging Markets", 272: "Emerging Markets", 428: "Emerging Markets",

    # Legal Regulatory (5)
    10: "Legal Regulatory", 104: "Legal Regulatory",
    291: "Legal Regulatory", 305: "Legal Regulatory",
    437: "Legal Regulatory",

    # Corporate Finance (4)
    140: "Corporate Finance", 193: "Corporate Finance",
    219: "Corporate Finance", 483: "Corporate Finance",

    # Labour (2)
    14: "Labour", 123: "Labour",

    # Corporate Leadership (2)
    379: "Corporate Leadership", 393: "Corporate Leadership",

    # Financial Markets (2)
    31: "Financial Markets", 167: "Financial Markets",

    # Capital Markets (1)
    233: "Capital Markets",

    # Consumer Affairs (3)
    20: "Consumer Affairs", 184: "Consumer Affairs", 236: "Consumer Affairs",

    # Foreign Policy Economics (2)
    334: "Foreign Policy Economics", 461: "Foreign Policy Economics",

    # Agriculture (2)
    9: "Agriculture", 120: "Agriculture",

    # Financial Regulation (1)
    147: "Financial Regulation",
}

# Apply assignments
for article_id, label in noise_assignments.items():
    mask = df_final['article_id'] == article_id
    df_final.loc[mask, 'semantic_label'] = label

# Save updated CSV
df_final.to_csv('business_subcategories_output.csv', index=False)

# Summary
remaining_noise = len(df_final[df_final['semantic_label'] == 'Noise'])
print("=" * 55)
print("      PHASE 8 — MANUAL NOISE CORRECTION")
print("=" * 55)
print(f" Manually assigned:     {len(noise_assignments)}")
print(f" Remaining noise:       {remaining_noise}")
print(f" Total classified:      {len(df_final[df_final['semantic_label'] != 'Noise'])}")
print(f"\n Final Label Distribution:")
print(df_final['semantic_label'].value_counts())
print("=" * 55)

      PHASE 8 — MANUAL NOISE CORRECTION
 Manually assigned:     97
 Remaining noise:       0
 Total classified:      510

 Final Label Distribution:
semantic_label
Macroeconomic Indicators    82
Corporate Governance        55
Crude Oil Markets           44
Telecommunications          42
Automotive Industry         31
Financial Markets           23
Foreign Exchange            20
Aviation Industry           20
Macroeconomics              19
Disaster Economics          16
Beverages Industry          15
Sports Business             13
Corporate Earnings          12
Energy Markets              11
Housing Market              11
Pension Funds               11
Pharmaceuticals             10
Mergers Acquisitions        10
International Trade         10
Asian Economics             10
Trade Policy                 8
Food Industry                8
Legal Regulatory             5
Corporate Finance            4
Oil Exploration              4
Consumer Affairs             3
Emerging Markets             3

EXERCISE 2: Named Entity Recognition with April Events Extraction

loaded spacy

In [12]:
import spacy
print(spacy.__version__)

# Check available models
import subprocess
result = subprocess.run(['python', '-m', 'spacy', 'info'], capture_output=True, text=True)
print(result.stdout)

3.8.14

============================== Info about spaCy ==============================

spaCy version    3.8.14                        
Location         c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\spacy
Platform         Windows-10-10.0.26200-SP0     
Python version   3.11.0                        
Pipelines        en_core_web_lg (3.8.0), en_core_web_sm (3.8.0)




### Load spaCy English Language Model

Loading the `en_core_web_sm` model which will be used to detect and classify named entities across all 510 business articles.

In [13]:
import spacy

# Check installed models
for model in ['en_core_web_sm', 'en_core_web_md', 'en_core_web_lg', 'en_core_web_trf']:
    try:
        nlp = spacy.load(model)
        print(f" {model} — available")
    except:
        print(f" {model} — not installed")

 en_core_web_sm — available
 en_core_web_md — not installed
 en_core_web_lg — available
 en_core_web_trf — not installed


## Load spaCy English Language Model
In this phase we verify and load spaCy's `en_core_web_lg` large English language model, which provides significantly better named entity recognition accuracy than the default `en_core_web_sm` model. The large model includes word vectors and improved NER capabilities essential for accurately identifying persons, organisations, and locations across BBC News articles.

In [14]:
import subprocess
result = subprocess.run(
    ['python', '-m', 'spacy', 'download', 'en_core_web_lg'],
    capture_output=True,
    text=True
)
print(result.stdout)
print(result.stderr)

     ---------------------------------------- 0.0/400.7 MB ? eta -:--:--
     ---------------------------------------- 0.0/400.7 MB ? eta -:--:--
     ---------------------------------------- 1.0/400.7 MB 5.0 MB/s eta 0:01:20
     ---------------------------------------- 2.1/400.7 MB 7.3 MB/s eta 0:00:55
      -------------------------------------- 6.0/400.7 MB 10.3 MB/s eta 0:00:39
      -------------------------------------- 8.7/400.7 MB 11.4 MB/s eta 0:00:35
     - ------------------------------------ 11.8/400.7 MB 11.5 MB/s eta 0:00:34
     - ------------------------------------ 14.4/400.7 MB 11.6 MB/s eta 0:00:34
     - ------------------------------------ 16.3/400.7 MB 11.4 MB/s eta 0:00:34
     - ------------------------------------ 18.1/400.7 MB 11.3 MB/s eta 0:00:34
     - ------------------------------------ 20.2/400.7 MB 10.8 MB/s eta 0:00:36
     -- ----------------------------------- 21.5/400.7 MB 10.5 MB/s eta 0:00:37
     -- ----------------------------------- 22.8/400.7

downloaded spacy libary

In [15]:
import spacy
nlp = spacy.load('en_core_web_lg')
print(f" en_core_web_lg loaded successfully!")
print(f"Pipeline: {nlp.pipe_names}")

 en_core_web_lg loaded successfully!
Pipeline: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


## Phase 9: Named Entity Recognition with April Events Extraction
In this phase we apply Named Entity Recognition (NER) to the **business category** (510 articles) using spaCy's `en_core_web_lg` model. NER is run on both `title` and `cleaned_text` combined for each article, extracting the following entities:

- **Named Persons** — individuals mentioned in the article, filtered using a false positive filter to remove movie titles, acronyms, and single-word misclassifications
- **Named Organisations** — companies, institutions, and bodies mentioned
- **Named Locations** — countries, cities, and regions mentioned
- **Person Roles** — each person classified by role (Politician, Business Executive, Central Banker, Expert/Analyst, TV/Film Personality, Musician, Sports Personality, Journalist, Other)
- **April Events** — sentences mentioning April extracted to satisfy the project's Desired requirement of identifying events that took place or were scheduled to take place in April

Results are saved to `business_ner_output.csv` with 439 articles containing named persons, 503 containing organisations, 488 containing locations, and 29 containing April events.

In [16]:
import pandas as pd

# Reload clean version without old NER columns
df = pd.read_csv('business_subcategories_output.csv')

# Drop old NER columns if they exist
cols_to_drop = ['named_persons', 'named_organisations', 'named_locations', 'person_roles', 'april_events']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

# Save clean version
df.to_csv('business_subcategories_output.csv', index=False)
print(f" Cleaned! Columns now: {df.columns.tolist()}")

 Cleaned! Columns now: ['file_name', 'article_id', 'title', 'cleaned_text', 'subcategory_id', 'semantic_label', 'confidence_score', 'article_keywords']


## Phase 9A: Named Entity Recognition — Persons, Organisations, Locations (Business Category)
In this phase we extract named entities from each article's combined `title` and `cleaned_text` using spaCy's `en_core_web_lg` model across all 510 business articles. Three entity types are extracted:

- **Named Persons** — validated through a false positive filter that removes movie titles, month names, acronyms, and single

In [6]:
# ============================================================
# PHASE 9A: NER — PERSONS, ORGS, LOCATIONS (BUSINESS)
# ============================================================
import spacy
import pandas as pd

nlp = spacy.load('en_core_web_lg')
df = pd.read_csv('business_subcategories_output.csv')
print(f" Loaded {len(df)} business articles")

# ============================================================
# FALSE POSITIVE FILTER
# ============================================================
FALSE_POSITIVES = {
    # Movie/TV titles
    'alexander', 'catwoman', 'rings', 'halo', 'titanic',
    'batman', 'superman', 'spiderman', 'matrix', 'avatar',
    'shrek', 'gladiator', 'braveheart', 'terminator',
    # Days/Months
    'monday', 'tuesday', 'wednesday', 'thursday', 'friday',
    'saturday', 'sunday', 'january', 'february', 'march',
    'april', 'may', 'june', 'july', 'august', 'september',
    'october', 'november', 'december',
    # Common misclassified words
    'lord', 'sir', 'mr', 'mrs', 'ms', 'dr', 'prof',
    'bertelsmann', 'aol', 'sec', 'gm', 'ibm', 'bbc',
    # Company/org names spaCy misclassifies as PERSON
    'allied domecq', "allied domecq's", 'parmalat', 'yukos',
    'enron', 'worldcom', 'andersen', 'hollinger',
    'christian aid', 'adam opel', 'adam opel -',
}

# Exact semantic_label values from business dataset
BUSINESS_LABELS = {
    'quarterly results', 'foreign exchange', 'crude oil', 'aviation industry',
    'beverages', 'asian economy', 'macroeconomics', 'foreign exchange markets',
    'agriculture', 'legal regulation', 'telecommunications', 'energy markets',
    'automotive industry', 'labour', 'financial markets', 'corporate governance',
    'international trade', 'retail sector'
}

def is_valid_person(name):
    if len(name) < 2:
        return False
    if name.lower() in FALSE_POSITIVES:
        return False
    if name.isupper():
        return False
    if not any(c.isalpha() for c in name):
        return False
    tokens = name.strip().split()
    if len(tokens) < 2:
        return False
    # filter possessives
    if name.endswith("'s") or name.endswith("s'"):
        return False
    # filter honorific-only entries
    honorifics = {'mr', 'mrs', 'ms', 'dr', 'prof', 'dame', 'sir', 'lord'}
    if tokens[0].lower().rstrip('.') in honorifics and len(tokens) == 2:
        return False
    # filter single initials like "J Smith"
    if len(tokens[0]) == 1 and tokens[0].isupper():
        return False
    # filter number entries
    if tokens[-1].isdigit():
        return False
    # filter comma-separated pairs
    if ',' in name:
        return False
    # filter entries with dots like "S. Young"
    if len(tokens[0]) <= 2 and '.' in tokens[0]:
        return False
    return True

# ============================================================
# KNOWN PERSON ROLE LOOKUP
# ============================================================
KNOWN_ROLES = {
    # Central Bankers
    'Alan Greenspan':           'Central Banker',
    # Politicians
    'Gordon Brown':             'Politician',
    'Peter Mandelson':          'Politician',
    'Prince Saud':              'Politician',
    'Ulrich Schulte-Strathaus': 'Politician',
    'Claude Veron-Reville':     'Politician',
    'Patricia Hewitt':          'Politician',
    'Anthony Gooch':            'Politician',
    'Erik Solheim':             'Politician',
    # Business Executives
    'Conrad Black':             'Business Executive',
    'Mikhail Khodorkovsky':     'Business Executive',
    'Rod Eddington':            'Business Executive',
    'Bruce Misamore':           'Business Executive',
    'Frank-Juergen Weise':      'Business Executive',
    'Alan Jones':               'Business Executive',
    # Expert/Analysts
    'Enrico Bondi':             'Expert/Analyst',
    'Henri Josserand':          'Expert/Analyst',
    'Paul Sheard':              'Expert/Analyst',
    'Mike Powell':              'Expert/Analyst',
    'Bhanu Baweja':             'Expert/Analyst',
    'Torao Toshitsune':         'Expert/Analyst',
    'Yuji Shimizu':             'Expert/Analyst',
    'Keith Rockwell':           'Expert/Analyst',
    'Helen Carroll':            'Expert/Analyst',
    'William Sharpe':           'Expert/Analyst',
    'Jason Daw':                'Expert/Analyst',
    'Kate Taylor':              'Expert/Analyst',
    # Lawyers
    'Letitia Clark':            'Lawyer',
    # Journalists
    'Jeremy Dear':              'Journalist',
    'Barry Fitzpatrick':        'Journalist',
    'Elizabeth Blunt':          'Journalist',
    # TV/Film
    'Jamie Firestone':          'TV/Film Personality',
    'Tim Osborne':              'TV/Film Personality',
}

# ============================================================
# PERSON ROLE CLASSIFIER
# ============================================================
def classify_role(person, context, semantic_label=''):
    # Check known persons first
    if person in KNOWN_ROLES:
        return KNOWN_ROLES[person]

    context = context.lower()
    semantic_label = semantic_label.lower().strip() if semantic_label else ''

    if any(w in context for w in [
        'president', 'prime minister', 'minister', 'senator', 'mp ',
        'chancellor', 'governor', 'secretary of state', 'politician',
        'parliament', 'elected', 'vote', 'constituency', 'opposition',
        'cabinet', 'treasury', 'labour party', 'conservative', 'democrat',
        'republican', 'union leader', 'trade union', 'general secretary'
    ]):
        return 'Politician'
    elif any(w in context for w in [
        'central bank', 'federal reserve', 'monetary policy',
        'interest rate', 'inflation target', 'bank of england',
        'european central bank', 'ecb', 'imf', 'world bank',
        'rate cut', 'rate hike', 'basis points'
    ]):
        return 'Central Banker'
    elif any(w in context for w in [
        'lawyer', 'attorney', 'legal counsel', 'defence counsel',
        'defendant', 'prosecution', 'lawsuit', 'sued', 'trial',
        'verdict', 'indicted', 'charged', 'acquitted', 'solicitor', 'barrister'
    ]):
        return 'Lawyer'
    elif any(w in context for w in [
        'actor', 'actress', 'director', 'film', 'movie', 'tv', 'television',
        'presenter', 'host', 'comedian', 'starred', 'cast', 'role', 'screen'
    ]):
        return 'TV/Film Personality'
    elif any(w in context for w in [
        'singer', 'musician', 'rapper', 'band', 'music', 'album',
        'song', 'artist', 'record', 'chart', 'tour', 'concert'
    ]):
        return 'Musician'
    elif any(w in context for w in [
        'ceo', 'chief executive', 'chairman', 'founder', 'executive',
        'managing director', 'mogul', 'tycoon', 'entrepreneur',
        'investor', 'billionaire', 'company', 'corporation',
        'chief financial', 'cfo', 'coo', 'board of directors',
        'shareholder', 'stake', 'merger', 'acquisition', 'takeover',
        'listed', 'stock', 'shares', 'profit', 'revenue', 'turnover',
        'subsidiary', 'conglomerate', 'holding company'
    ]):
        return 'Business Executive'
    elif any(w in context for w in [
        'analyst', 'economist', 'researcher', 'professor', 'scientist',
        'expert', 'academic', 'lecturer', 'scholar', 'strategy',
        'forecast', 'prediction', 'think tank', 'institute', 'consultant',
        'adviser', 'advisor', 'specialist', 'commentator on'
    ]):
        return 'Expert/Analyst'
    elif any(w in context for w in [
        'coach', 'manager', 'player', 'footballer', 'athlete',
        'champion', 'scored', 'match', 'tournament', 'league',
        'olympic', 'medal', 'world record', 'championship'
    ]):
        return 'Sports Personality'
    elif any(w in context for w in [
        'journalist', 'reporter', 'editor', 'correspondent', 'critic',
        'columnist', 'broadcaster', 'commentator', 'pundit', 'anchor',
        'newspaper', 'magazine', 'media', 'press'
    ]):
        return 'Journalist'
    # Subcategory fallback — business labels default to Business Executive
    elif semantic_label in BUSINESS_LABELS:
        return 'Business Executive'
    else:
        return 'Other'

# ============================================================
# NER EXTRACTION FUNCTION
# ============================================================
def extract_persons_orgs_locations(title, cleaned_text, semantic_label=''):
    combined = f"{title}. {cleaned_text}"
    doc = nlp(combined[:100000])

    persons = []
    person_roles = []
    orgs = []
    locations = []

    seen_persons = set()
    seen_orgs = set()
    seen_locations = set()

    for ent in doc.ents:

        if ent.label_ == 'PERSON':
            if ent.text not in seen_persons and is_valid_person(ent.text):
                seen_persons.add(ent.text)
                start = max(0, ent.start_char - 300)
                end = min(len(combined), ent.end_char + 300)
                context = combined[start:end]
                role = classify_role(ent.text, context, semantic_label)
                persons.append(ent.text)
                person_roles.append(f"{ent.text}: {role}")

        elif ent.label_ == 'ORG':
            if ent.text not in seen_orgs:
                seen_orgs.add(ent.text)
                orgs.append(ent.text)

        elif ent.label_ in ['GPE', 'LOC']:
            if ent.text not in seen_locations:
                seen_locations.add(ent.text)
                locations.append(ent.text)

    return {
        'named_persons':       ' | '.join(persons),
        'named_organisations': ' | '.join(orgs),
        'named_locations':     ' | '.join(locations),
        'person_roles':        ' | '.join(person_roles),
    }

# ============================================================
# RUN NER ON ALL ARTICLES
# ============================================================
print(" Running NER (Persons, Orgs, Locations)...")

named_persons       = []
named_organisations = []
named_locations     = []
person_roles        = []

for i, row in df.iterrows():
    result = extract_persons_orgs_locations(
        str(row['title']),
        str(row['cleaned_text']),
        str(row.get('semantic_label', ''))
    )
    named_persons.append(result['named_persons'])
    named_organisations.append(result['named_organisations'])
    named_locations.append(result['named_locations'])
    person_roles.append(result['person_roles'])

    if (i + 1) % 50 == 0:
        print(f"   Processed {i + 1}/{len(df)} articles...")

df['named_persons']       = named_persons
df['named_organisations'] = named_organisations
df['named_locations']     = named_locations
df['person_roles']        = person_roles

print(f"\n Person/Org/Location NER complete!")
print(f" Articles with Persons:   {sum(1 for x in named_persons if x)}")
print(f" Articles with Orgs:      {sum(1 for x in named_organisations if x)}")
print(f" Articles with Locations: {sum(1 for x in named_locations if x)}")

# ============================================================
# AUDIT — check for remaining 'Other'
# ============================================================
other_count = df['person_roles'].str.contains('Other', na=False).sum()
print(f"\n Articles still containing 'Other': {other_count}")

other_rows = df[df['person_roles'].str.contains("Other", na=False)].copy()

def extract_others(person_roles_str):
    entries = person_roles_str.split(" | ")
    return [e.strip() for e in entries if e.strip().endswith(": Other")]

other_rows['other_persons'] = other_rows['person_roles'].apply(extract_others)

for _, row in other_rows.iterrows():
    print(f"Article {row['article_id']} | {row['title'][:60]}")
    for person in row['other_persons']:
        print(f"   --> {person}")
    print()

 Loaded 510 business articles
 Running NER (Persons, Orgs, Locations)...
   Processed 50/510 articles...
   Processed 100/510 articles...
   Processed 150/510 articles...
   Processed 200/510 articles...
   Processed 250/510 articles...
   Processed 300/510 articles...
   Processed 350/510 articles...
   Processed 400/510 articles...
   Processed 450/510 articles...
   Processed 500/510 articles...

 Person/Org/Location NER complete!
 Articles with Persons:   433
 Articles with Orgs:      503
 Articles with Locations: 488

 Articles still containing 'Other': 0


## Phase 9B: April Events Extraction (Business Category)
In this phase we extract April-related events from each article using spaCy's DATE entity recognition. For every article where a DATE entity containing "April" is detected in the combined `title` and `cleaned_text`, a 300-character contextual window surrounding the mention is captured as the event summary. This satisfies the project's Desired requirement of identifying events that took place or were scheduled to take place in April. Results are saved to `business_ner_output.csv` with **29 out of 510** business articles containing April events.

In [7]:
# ============================================================
# PHASE 9B: NER — APRIL EVENTS (BUSINESS)
# ============================================================

def extract_april_events(title, cleaned_text):
    combined = f"{title}. {cleaned_text}"
    doc = nlp(combined[:100000])

    april_events = []

    for ent in doc.ents:
        if ent.label_ == 'DATE':
            if 'april' in ent.text.lower():
                start = max(0, ent.start_char - 100)
                end = min(len(combined), ent.end_char + 200)
                april_events.append(combined[start:end].strip())

    return ' | '.join(april_events) if april_events else ''

print(" Extracting April events...")

april_events = []
for i, row in df.iterrows():
    result = extract_april_events(str(row['title']), str(row['cleaned_text']))
    april_events.append(result)

    if (i + 1) % 50 == 0:
        print(f"   Processed {i + 1}/{len(df)} articles...")

df['april_events'] = april_events

print(f"\n April Events extraction complete!")
print(f"  Articles with April Events: {sum(1 for x in april_events if x)}")

# Save final output
df.to_csv('business_ner_output.csv', index=False)
print(f"\n Saved to business_ner_output.csv")

print("\n" + "=" * 55)
print("      BUSINESS NER — FINAL SUMMARY")
print("=" * 55)
print(f" Total Articles:             {len(df)}")
print(f" Articles with Persons:      {sum(1 for x in named_persons if x)}")
print(f" Articles with Orgs:         {sum(1 for x in named_organisations if x)}")
print(f" Articles with Locations:    {sum(1 for x in named_locations if x)}")
print(f"  Articles with April Events: {sum(1 for x in april_events if x)}")
print("=" * 55)

 Extracting April events...
   Processed 50/510 articles...
   Processed 100/510 articles...
   Processed 150/510 articles...
   Processed 200/510 articles...
   Processed 250/510 articles...
   Processed 300/510 articles...
   Processed 350/510 articles...
   Processed 400/510 articles...
   Processed 450/510 articles...
   Processed 500/510 articles...

 April Events extraction complete!
  Articles with April Events: 29

 Saved to business_ner_output.csv

      BUSINESS NER — FINAL SUMMARY
 Total Articles:             510
 Articles with Persons:      433
 Articles with Orgs:         503
 Articles with Locations:    488
  Articles with April Events: 29


## Phase 9C: Save Complete NER Output (Business Category)
In this phase we consolidate all extracted NER columns — `named_persons`, `named_organisations`, `named_locations`, `person_roles`, and `april_events` — into the final dataframe and save to `business_ner_output.csv`. A sample verification of the first article is printed to confirm all columns are correctly populated before proceeding to the next category.

In [8]:
# ============================================================
# Phase 9C SAVE COMPLETE NER OUTPUT
# ============================================================

df['named_persons']       = named_persons
df['named_organisations'] = named_organisations
df['named_locations']     = named_locations
df['person_roles']        = person_roles
df['april_events']        = april_events

# Save with ALL columns
df.to_csv('business_ner_output.csv', index=False)
print(" Saved to business_ner_output.csv")
print(f" Columns: {df.columns.tolist()}")
print(f"\n Sample row 1:")
print(f" Persons:  {df['named_persons'].iloc[0]}")
print(f" Orgs:     {df['named_organisations'].iloc[0]}")
print(f" Location: {df['named_locations'].iloc[0]}")
print(f" Roles:    {df['person_roles'].iloc[0]}")
print(f"  April:    {df['april_events'].iloc[0]}")

 Saved to business_ner_output.csv
 Columns: ['file_name', 'article_id', 'title', 'subcategory_id', 'semantic_label', 'confidence_score', 'article_keywords', 'cleaned_text', 'named_persons', 'named_organisations', 'named_locations', 'person_roles', 'april_events']

 Sample row 1:
 Persons:  Richard Parsons
 Orgs:     Time Warner | TimeWarner | USD 639m year-earlier | Google | Warner Bros | AOL | the US Securities Exchange Commission | SEC | Time Warner's | Rings | AOL Europe
 Location: US
 Roles:    Richard Parsons: Business Executive
  April:    
